In [1]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset (スケール統一版) --------
class ScaledModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path):
                continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0:
                continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15:
                continue

            own_speeds = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed_seq = t - s
                rel_speed_feat = rel_speed_seq[:14]
                rel_speed_feat = np.pad(rel_speed_feat, (0, 1), mode='constant')

                rel_speed = np.mean(rel_speed_seq)

                # -------- 特徴量作成 --------
                feature = np.concatenate([
                    modes, d, s, a, s1, d1, d2, rel_acc, rel_speed_feat, *d_smooths
                ])

                # -------- 特徴量補正（スケール統一） --------
                feature = np.clip(feature, -300, 300)
                feature_min = feature.min()
                feature_max = feature.max()
                feature = (feature - feature_min) / (feature_max - feature_min + 1e-8)

                # -------- ターゲット補正（スケール統一） --------
                rel_speed = np.clip(rel_speed, -30, 30)
                rel_speed = (rel_speed + 30) / 60.0

                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid

# -------- Collate --------
def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)

# -------- モデル --------
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_dim=15, feature_dim=15, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)

        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, F = x.shape
        x = x.view(B, 15, -1)
        x = self.pre_fc(x.reshape(-1, x.size(2))).view(B, 15, -1)

        lstm_out, _ = self.lstm(x)

        attn_scores = self.attn_fc(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)

        out = self.dropout(context)
        return self.fc_out(out).squeeze(1)

# -------- 学習ループ --------
def train_lstm_model(dataset, save_path="model_lstm_attn_scaled.pth"):
    scenes = sorted(list(set([item[-1] for item in dataset.items])))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx[:6000])
    val_ds = Subset(dataset, val_idx[:1500])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim = train_ds[0][0].shape[0] // 15
    model = ExtendedLSTMWithAttention(input_dim=15, feature_dim=input_dim).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 30
    patience_counter = 0

    for epoch in range(200):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step()

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行部分 --------
if __name__ == "__main__":
    crop_root = "../train_retry/train_crops"
    annot_root = "../train/train_annotations"
    distance_json_path = "../train_retry/trainestimates2.json"

    dataset = ScaledModeAndFeatureDataset(
        crop_root=crop_root,
        annot_root=annot_root,
        distance_json_path=distance_json_path,
        max_items=7500
    )

    model = train_lstm_model(dataset, save_path="model_lstm_attn_scaled.pth")


[Train 1]: 100%|██████████| 93/93 [00:00<00:00, 105.80it/s]


Epoch 1 | Train Loss: 0.0332 | Val Loss: 0.0031
✅ Saved model to model_lstm_attn_scaled.pth (val_loss=0.0031)


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 200.64it/s]


Epoch 2 | Train Loss: 0.0044 | Val Loss: 0.0022
✅ Saved model to model_lstm_attn_scaled.pth (val_loss=0.0022)


[Train 3]: 100%|██████████| 93/93 [00:00<00:00, 205.97it/s]


...

Epoch 92 | Train Loss: 0.0004 | Val Loss: 0.0004


[Train 93]: 100%|██████████| 93/93 [00:00<00:00, 194.78it/s]


Epoch 93 | Train Loss: 0.0004 | Val Loss: 0.0004


[Train 94]: 100%|██████████| 93/93 [00:00<00:00, 200.91it/s]


Epoch 94 | Train Loss: 0.0004 | Val Loss: 0.0004
🛑 Early stopping at epoch 94


In [2]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
import re

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ---------- フレーム番号抽出 ----------
def extract_frame_number(filename):
    match = re.search(r'frame_(\d+)\.png', filename)
    return int(match.group(1)) if match else -1

# ---------- 推論用データセット ----------
class InferenceDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        self.scene_ids = sorted(os.listdir(crop_root))
        for sid in self.scene_ids:
            if not sid.isdigit():
                continue

            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path):
                continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")],
                           key=extract_frame_number)
            if len(files) < 15:
                continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))

            own_speeds = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed_feat = np.zeros(15, dtype=np.float32)

                frame_number = extract_frame_number(files[i + 14])  # 中心フレーム番号

                feature = np.concatenate([
                    modes, d, s, a, s1, d1, d2, rel_acc, rel_speed_feat, *d_smooths
                ])

                self.items.append((feature.astype(np.float32), sid, frame_number))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, sid, frame_number = self.items[idx]
        return torch.tensor(feature), sid, frame_number

# ---------- モデル定義 ----------
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_dim=15, feature_dim=15, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)
        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, F = x.shape
        x = x.view(B, 15, -1)
        x = self.pre_fc(x.reshape(-1, x.size(2))).view(B, 15, -1)
        lstm_out, _ = self.lstm(x)
        attn_scores = self.attn_fc(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        out = self.dropout(context)
        return self.fc_out(out).squeeze(1)

# ---------- 推論実行 ----------
def inference(model_path, crop_root, annot_root, distance_json_path, output_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = InferenceDataset(crop_root, annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    input_dim = dataset[0][0].shape[0] // 15
    model = ExtendedLSTMWithAttention(input_dim=15, feature_dim=input_dim)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    predictions = {}
    own_speeds_all = {}
    scene_to_len = {}

    # アノテーションから各シーンのフレーム数とOwnSpeed取得
    for sid in sorted(os.listdir(annot_root)):
        if sid.endswith(".json"):
            with open(os.path.join(annot_root, sid), encoding="utf-8") as f:
                ann = json.load(f)
            scene_id = sid.replace(".json", "")
            scene_to_len[scene_id] = len(ann['sequence'])
            own_speeds_all[scene_id] = [frame['OwnSpeed'] for frame in ann['sequence']]

    with torch.no_grad():
        for feats, sids, frame_numbers in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()

            for pred, sid, frame_number in zip(preds, sids, frame_numbers):
                sid = sid
                frame_number = int(frame_number)
                if sid not in predictions:
                    predictions[sid] = []

                while len(predictions[sid]) < frame_number - 1:
                    predictions[sid].append(0.0)

                if frame_number < 20:
                    predictions[sid].append(0.0)
                else:
                    own_speed = own_speeds_all[sid][frame_number - 1]
                    tgt_speed_pred = float(pred) + own_speed
                    predictions[sid].append(tgt_speed_pred)

    for sid in scene_to_len:
        total_len = scene_to_len[sid]
        if sid not in predictions:
            predictions[sid] = [0.0] * total_len
        else:
            while len(predictions[sid]) < total_len:
                predictions[sid].append(0.0)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    print(f"\u2705 推論完了: {output_path} に保存しました")

# ---------- 実行部 ----------
if __name__ == "__main__":
    model_path = "model_lstm_attn_scaled.pth"
    crop_root = "../test_retry/test_crops"
    annot_root = "../test/test_annotations"
    distance_json_path = "../test_retry/testestimates_smoothed.json"
    output_path = "submission.json"
    inference(model_path, crop_root, annot_root, distance_json_path, output_path)

100%|██████████| 414/414 [00:01<00:00, 381.38it/s]


✅ 推論完了: submission.json に保存しました
